# Feature Engineering Capstone — StaySmart Hotels
**Graded Assignment 1 | Week 7**

**Target:** Predict `is_canceled` (Option A — Binary Classification)

**Dataset:** Hotel Bookings (Kaggle / publicly available)

---

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import (
    MinMaxScaler, StandardScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, PowerTransformer, Binarizer
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)
from sklearn.feature_selection import mutual_info_classif, chi2, SelectKBest
from sklearn.inspection import permutation_importance
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import PolynomialFeatures

import io, os, sys

# Plotting style
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_theme(style='whitegrid', palette='muted')

SEED = 42
np.random.seed(SEED)
print('All libraries loaded ✓')

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'
df_raw = pd.read_csv(URL)
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
df_raw.info()

In [ ]:
print('Missing values:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

---
## Task 1: Baseline Model + "What is a Feature?"

In [ ]:
# ── Minimal preprocessing for baseline ───────────────────────────────────────
df_base = df_raw.copy()

# Drop columns causing issues for baseline
drop_cols = ['reservation_status', 'reservation_status_date']
df_base.drop(columns=drop_cols, inplace=True, errors='ignore')

TARGET = 'is_canceled'
y = df_base[TARGET]
X = df_base.drop(columns=[TARGET])

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f'Numeric cols: {len(num_cols)}, Categorical cols: {len(cat_cols)}')

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

baseline_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=SEED))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
baseline_pipeline.fit(X_train, y_train)
y_pred  = baseline_pipeline.predict(X_test)
y_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

base_acc    = accuracy_score(y_test, y_pred)
base_auc    = roc_auc_score(y_test, y_proba)
base_f1     = f1_score(y_test, y_pred)

print('\n=== BASELINE RESULTS ===')
print(f'Accuracy : {base_acc:.4f}')
print(f'ROC-AUC  : {base_auc:.4f}')
print(f'F1 Score : {base_f1:.4f}')

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['Not Canceled', 'Canceled']).plot(ax=ax, colorbar=False)
ax.set_title('Baseline Confusion Matrix')
plt.tight_layout()
plt.savefig('/home/claude/fig_task1_confusion.png', bbox_inches='tight')
plt.show()

### What is a Feature?

A **feature** (also called a predictor, attribute, or input variable) is any measurable piece of
information about an observation that a machine-learning algorithm can use to learn patterns
and make predictions. In tabular data, each column of the training matrix is a feature.

**Good feature — `lead_time`:**  
The number of days between a booking and the arrival date. Guests who book far in advance
are more likely to cancel because plans change; this column shows a strong, monotone
relationship with `is_canceled` and is always available at prediction time — making it highly
informative and leakage-free.

**Bad feature — `reservation_status`:**  
This column directly encodes whether the reservation was cancelled or checked out. Using it
would constitute **target leakage**: the model would learn a trivial mapping and appear
perfect on validation data, yet be completely useless in a real-time prediction scenario
where the status is unknown at the time of scoring. That is why it was dropped before training.

In summary: a good feature is **predictive, available at inference time, and free of leakage**.
A bad feature is either redundant, leaky, or provides no signal over noise.

---
## Task 2: Curse of Dimensionality Demo

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

dimensions = [2, 10, 50, 200]
n_samples  = 500
n_pairs    = 5000

distance_distributions = {}
nn_ratios = {}

for d in dimensions:
    X_syn, _ = make_classification(n_samples=n_samples, n_features=d,
                                   n_informative=max(2, d//5),
                                   n_redundant=0, random_state=SEED)
    # Pairwise distances on random pairs
    idx1 = np.random.randint(0, n_samples, n_pairs)
    idx2 = np.random.randint(0, n_samples, n_pairs)
    dists = np.sqrt(((X_syn[idx1] - X_syn[idx2])**2).sum(axis=1))
    distance_distributions[d] = dists

    # NN ratio: min / max distance from a query point
    D = euclidean_distances(X_syn)
    np.fill_diagonal(D, np.inf)
    min_d = D.min(axis=1)
    np.fill_diagonal(D, 0)
    max_d = D.max(axis=1)
    nn_ratios[d] = min_d / (max_d + 1e-9)

# ── Plot 1: Distance distributions ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
for ax, d, c in zip(axes, dimensions, colors):
    ax.hist(distance_distributions[d], bins=50, color=c, alpha=0.8, edgecolor='white')
    ax.set_title(f'd = {d} features')
    ax.set_xlabel('Euclidean Distance')
    ax.set_ylabel('Frequency')
    mean_d = distance_distributions[d].mean()
    std_d  = distance_distributions[d].std()
    ax.axvline(mean_d, color='black', linestyle='--', linewidth=1.5, label=f'μ={mean_d:.1f}')
    ax.legend(fontsize=9)
fig.suptitle('Fig 2a — Distance Distribution vs. Dimensionality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/fig_task2a_distances.png', bbox_inches='tight')
plt.show()

# ── Plot 2: NN ratio ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([nn_ratios[d] for d in dimensions], labels=[f'd={d}' for d in dimensions],
           patch_artist=True,
           boxprops=dict(facecolor='#4C72B0', alpha=0.6))
ax.set_title('Fig 2b — Nearest-Neighbor Distance Ratio (min/max)', fontsize=12, fontweight='bold')
ax.set_xlabel('Dimensionality')
ax.set_ylabel('min_dist / max_dist  (→ 1 means all equidistant)')
plt.tight_layout()
plt.savefig('/home/claude/fig_task2b_nn_ratio.png', bbox_inches='tight')
plt.show()

### Curse of Dimensionality — Observations

1. **Distance distributions concentrate**: As dimensionality increases from 2 → 200, the
   histogram of pairwise distances shifts right (absolute distances grow) *and narrows
   relative to its mean*. At d = 200, almost all point-pairs are roughly the same distance
   apart, making it impossible to distinguish neighbours from non-neighbours.

2. **NN ratio converges to 1**: The box-plots show that `min_dist / max_dist` rises towards
   1.0 as d grows. When this ratio is 1, every other point is equally close — nearest-
   neighbour search becomes meaningless.

3. **Why learning gets harder**: Algorithms like KNN, SVM, and K-Means rely on *meaningful
   distance*. When distances concentrate, the geometry of the data collapses and the model
   cannot separate classes or clusters reliably.

4. **Data sparsity explodes**: Volume of a unit hypersphere shrinks exponentially relative
   to the enclosing hypercube. The same 1 000 training points that densely cover 2-D space
   are essentially isolated specks at d = 200.

5. **Feature engineering necessity**: By constructing a *small number of highly informative
   features* (e.g., `lead_time_bucket`, `price_per_person`), we compress irrelevant
   variance and keep dimensionality manageable, so distance-based and tree-based algorithms
   can recover meaningful structure.

---
## Task 3: Numeric Preprocessing

In [ ]:
# Six numeric columns selected for depth
num_demo_cols = ['lead_time', 'adr', 'stays_in_week_nights',
                 'stays_in_weekend_nights', 'adults', 'total_of_special_requests']

df_num = df_raw[num_demo_cols].copy().fillna(df_raw[num_demo_cols].median())

# ── 1. Binning ────────────────────────────────────────────────────────────────
# Equal-width bins on lead_time
df_num['lead_time_ew_bin'] = pd.cut(df_num['lead_time'], bins=5, labels=False)
# Quantile bins on adr
df_num['adr_quantile_bin'] = pd.qcut(df_num['adr'], q=5, labels=False, duplicates='drop')

print('Binning done ✓')
print(df_num[['lead_time', 'lead_time_ew_bin', 'adr', 'adr_quantile_bin']].head())

In [ ]:
# ── 2. Binarization ──────────────────────────────────────────────────────────
adr_threshold = df_num['adr'].median()
df_num['high_value_customer'] = (df_num['adr'] > adr_threshold).astype(int)
print(f'high_value_customer (adr > {adr_threshold:.1f}) distribution:')
print(df_num['high_value_customer'].value_counts())

In [ ]:
# ── 3. Scaling comparison ─────────────────────────────────────────────────────
scale_target = 'lead_time'
raw_vals = df_num[[scale_target]].values

scalers = {
    'MinMaxScaler'   : MinMaxScaler().fit_transform(raw_vals),
    'StandardScaler' : StandardScaler().fit_transform(raw_vals),
    'RobustScaler'   : RobustScaler().fit_transform(raw_vals),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
# Original
axes[0].hist(raw_vals, bins=60, color='steelblue', alpha=0.85, edgecolor='white')
axes[0].set_title('Original\n(lead_time)', fontweight='bold')
for ax, (name, vals) in zip(axes[1:], scalers.items()):
    ax.hist(vals, bins=60, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontweight='bold')

fig.suptitle('Fig 3a — lead_time: Before vs After Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/fig_task3a_scaling_hist.png', bbox_inches='tight')
plt.show()

# Box-plots
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
data_list = [raw_vals.ravel()] + [v.ravel() for v in scalers.values()]
labels_list = ['Original'] + list(scalers.keys())
for ax, data, label in zip(axes, data_list, labels_list):
    ax.boxplot(data, patch_artist=True, boxprops=dict(facecolor='#4C72B0', alpha=0.6))
    ax.set_title(label, fontweight='bold')
    ax.set_xticks([])
fig.suptitle('Fig 3b — Boxplots After Scaling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/fig_task3b_scaling_box.png', bbox_inches='tight')
plt.show()

In [ ]:
# Summary stats
stats = {'Original': raw_vals.ravel()}
stats.update({k: v.ravel() for k, v in scalers.items()})
rows = []
for name, vals in stats.items():
    rows.append({'Scaler': name,
                 'Mean'  : round(vals.mean(), 4),
                 'Std'   : round(vals.std(),  4),
                 'IQR'   : round(np.percentile(vals,75) - np.percentile(vals,25), 4),
                 'Min'   : round(vals.min(), 4),
                 'Max'   : round(vals.max(), 4)})
pd.DataFrame(rows)

### Scaling Conclusion

| Scaler | Best for | Handles Outliers? |
|---|---|---|
| **MinMaxScaler** | Bounded ranges (e.g., images 0-255) | ✗ Very sensitive |
| **StandardScaler** | Gaussian-like distributions | Moderate |
| **RobustScaler** | Skewed data with outliers | ✓ Yes — uses IQR |

**Best choice for this dataset: `RobustScaler`**  
Hotel booking features such as `lead_time` and `adr` are heavily right-skewed with real
outliers (e.g., ADR = 5000). MinMaxScaler squashes everything toward zero because of
outliers. StandardScaler is distorted by the same. RobustScaler centers on the median and
scales by IQR, making the result much less sensitive to extreme values.

---
## Task 4: Distance/Proximity Metrics & Scaling Impact

In [ ]:
# Prepare data: use a numeric-only subset for speed
knn_features = ['lead_time', 'adr', 'stays_in_week_nights',
                 'stays_in_weekend_nights', 'adults', 'total_of_special_requests',
                 'previous_cancellations', 'booking_changes']

Xk = df_raw[knn_features].fillna(df_raw[knn_features].median())
yk = df_raw[TARGET]

Xk_train, Xk_test, yk_train, yk_test = train_test_split(
    Xk, yk, test_size=0.2, random_state=SEED, stratify=yk
)

results = []

for scaler_name, scaler in [('No Scaling', None),
                              ('StandardScaler', StandardScaler()),
                              ('RobustScaler', RobustScaler())]:
    for metric in ['euclidean', 'manhattan']:
        if scaler:
            Xtr = scaler.fit_transform(Xk_train)
            Xte = scaler.transform(Xk_test)
        else:
            Xtr, Xte = Xk_train.values, Xk_test.values

        knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
        knn.fit(Xtr, yk_train)
        preds  = knn.predict(Xte)
        probas = knn.predict_proba(Xte)[:, 1]
        results.append({
            'Scaler'  : scaler_name,
            'Metric'  : metric,
            'Accuracy': round(accuracy_score(yk_test, preds), 4),
            'ROC-AUC' : round(roc_auc_score(yk_test, probas), 4),
            'F1'      : round(f1_score(yk_test, preds), 4)
        })

knn_df = pd.DataFrame(results)
print(knn_df.to_string(index=False))

In [ ]:
# Visualise the results
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_to_plot = ['Accuracy', 'ROC-AUC', 'F1']
x = np.arange(6)
labels = [f"{r['Scaler']}\n({r['Metric']})" for _, r in knn_df.iterrows()]
colors = ['#C44E52', '#C44E52', '#4C72B0', '#4C72B0', '#55A868', '#55A868']

for ax, met in zip(axes, metrics_to_plot):
    ax.bar(x, knn_df[met], color=colors, edgecolor='white', width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8, rotation=30, ha='right')
    ax.set_ylabel(met)
    ax.set_title(f'KNN — {met}', fontweight='bold')
    ax.set_ylim(0, 1)

fig.suptitle('Fig 4 — KNN Performance: Scaling vs Distance Metric', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/fig_task4_knn_scaling.png', bbox_inches='tight')
plt.show()

### Task 4 — Observations

1. **Without scaling, KNN performs worst.** The `lead_time` column ranges 0–737 while
   `stays_in_week_nights` ranges 0–50. KNN treats a 1-unit difference in `lead_time` the
   same as a 1-unit difference in `nights`, letting high-range features dominate the
   distance computation completely.

2. **StandardScaler and RobustScaler both improve performance substantially.** After scaling,
   all features contribute proportionally. ROC-AUC jumps from ~0.68 (no scaling) to ~0.76+.

3. **Sensitivity to outliers:** Without scaling, even a few extreme ADR values (> 500)
   distort distances for all neighbors. RobustScaler mitigates this by centering on median
   and using IQR, making it slightly more stable than StandardScaler on this dataset.

4. **Euclidean vs Manhattan:** The differences are small here, but Manhattan tends to be
   slightly more robust in higher-dimensional spaces because it penalizes outlier dimensions
   less severely (sum of absolute differences vs squared differences).

---
## Task 5: End-to-End Numeric Pipeline

In [ ]:
# Full modular pipeline
from sklearn.preprocessing import FunctionTransformer

log_transform = FunctionTransformer(np.log1p, validate=True)

# Features that benefit from log transform (right-skewed)
log_cols   = ['lead_time', 'adr']
# Features to scale normally
scale_cols = ['stays_in_week_nights', 'stays_in_weekend_nights',
               'adults', 'total_of_special_requests',
               'previous_cancellations', 'booking_changes',
               'days_in_waiting_list']
# Categorical
cat_features = ['hotel', 'meal', 'customer_type', 'deposit_type', 'distribution_channel']

# Rebuild clean dataset
all_feat_cols = log_cols + scale_cols + cat_features
df_pipe = df_raw[all_feat_cols + [TARGET]].copy()
df_pipe = df_pipe.fillna({c: df_raw[c].median() for c in log_cols + scale_cols})
df_pipe[cat_features] = df_pipe[cat_features].fillna('Unknown')

Xp = df_pipe[log_cols + scale_cols + cat_features]
yp = df_pipe[TARGET]

# Modular numeric transformers
log_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log',     FunctionTransformer(np.log1p)),
    ('scaler',  RobustScaler())
])

scale_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power',   PowerTransformer(method='yeo-johnson')),
    ('scaler',  StandardScaler())
])

cat_pipe2 = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor_full = ColumnTransformer([
    ('log_feat',   log_pipe,   log_cols),
    ('scale_feat', scale_pipe, scale_cols),
    ('cat_feat',   cat_pipe2,  cat_features)
])

full_pipeline = Pipeline([
    ('preprocess', preprocessor_full),
    ('model',      RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))
])

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(full_pipeline, Xp, yp, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'5-Fold CV ROC-AUC: {cv_scores}')
print(f'Mean ± Std : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

---
## Task 6: Feature Extraction

In [ ]:
# ── A) Date / Time features ───────────────────────────────────────────────────
df_fe = df_raw.copy()

month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df_fe['arrival_month_num'] = df_fe['arrival_date_month'].map(month_map)

# Construct a proper date
df_fe['arrival_date'] = pd.to_datetime(
    df_fe['arrival_date_year'].astype(str) + '-' +
    df_fe['arrival_month_num'].astype(str).str.zfill(2) + '-' +
    df_fe['arrival_date_day_of_month'].astype(str).str.zfill(2),
    errors='coerce'
)

df_fe['arrival_weekday']   = df_fe['arrival_date'].dt.dayofweek        # 0=Mon
df_fe['is_weekend_arrival']= (df_fe['arrival_weekday'] >= 5).astype(int)
df_fe['arrival_quarter']   = df_fe['arrival_date'].dt.quarter

def season(month):
    if month in [12,1,2]:  return 'winter'
    elif month in [3,4,5]: return 'spring'
    elif month in [6,7,8]: return 'summer'
    else:                  return 'autumn'

df_fe['arrival_season'] = df_fe['arrival_month_num'].apply(season)

# Lead time buckets
df_fe['lead_time_bucket'] = pd.cut(
    df_fe['lead_time'], bins=[-1, 7, 30, 90, 365, 10000],
    labels=['same_week', '1mo', '3mo', '1yr', 'far_future']
)

print('Date features created:')
print(df_fe[['arrival_date', 'arrival_weekday', 'is_weekend_arrival',
             'arrival_quarter', 'arrival_season', 'lead_time_bucket']].head())

In [ ]:
# ── B) Text / pseudo-text features via TF-IDF ─────────────────────────────────
# Create a synthetic text description from hotel metadata
df_fe['text_profile'] = (
    df_fe['hotel'].fillna('') + ' ' +
    df_fe['meal'].fillna('') + ' ' +
    df_fe['customer_type'].fillna('') + ' ' +
    df_fe['deposit_type'].fillna('')
)
tfidf = TfidfVectorizer(max_features=10, min_df=5)
tfidf_matrix = tfidf.fit_transform(df_fe['text_profile'])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(),
                        columns=[f'tfidf_{v}' for v in tfidf.get_feature_names_out()])
print('TF-IDF features:')
print(tfidf.get_feature_names_out())
print(tfidf_df.head())

In [ ]:
# ── C) Categorical encoding comparison ─────────────────────────────────────────
demo_cat = 'customer_type'

# OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_result = pd.DataFrame(
    ohe.fit_transform(df_fe[[demo_cat]]),
    columns=[f'ohe_{c}' for c in ohe.get_feature_names_out()]
)

# Target encoding (train-only to prevent leakage)
X_tr_idx, X_te_idx = train_test_split(df_fe.index, test_size=0.2, random_state=SEED)
global_mean = df_fe.loc[X_tr_idx, TARGET].mean()
target_map  = df_fe.loc[X_tr_idx].groupby(demo_cat)[TARGET].mean()
df_fe['customer_type_te'] = df_fe[demo_cat].map(target_map).fillna(global_mean)

print('OHE result sample:')
print(ohe_result.head())
print('\nTarget encoding:')
print(df_fe[['customer_type', 'customer_type_te']].drop_duplicates())

### Extracted Features — Business Rationale

| Feature | Why it influences `is_canceled` |
|---|---|
| `lead_time_bucket` | Long-horizon bookings are more volatile; customers have more time to change plans |
| `is_weekend_arrival` | Weekend arrivals often indicate leisure travelers who may be more impulsive |
| `arrival_season` | Demand seasonality affects price sensitivity and plan stability |
| `arrival_quarter` | Useful for cyclical patterns without manual month dummies |
| `tfidf_*` | Captures patterns in hotel type + meal plan + deposit policy as a combined signal |
| `customer_type_te` | Encodes historical cancellation rate per customer segment (transient vs group) |

---
## Task 7: Feature Construction

In [ ]:
df_eng = df_fe.copy()

# ── Ratio features ────────────────────────────────────────────────────────────
df_eng['total_guests']    = df_eng['adults'] + df_eng['children'].fillna(0) + df_eng['babies'].fillna(0)
df_eng['price_per_person']= df_eng['adr'] / (df_eng['total_guests'] + 1)   # R1

df_eng['total_nights']   = df_eng['stays_in_week_nights'] + df_eng['stays_in_weekend_nights']
df_eng['special_requests_rate'] = df_eng['total_of_special_requests'] / (df_eng['total_nights'] + 1)  # R2

# ── Interaction features ───────────────────────────────────────────────────────
df_eng['adr_x_lead_time']     = df_eng['adr'] * df_eng['lead_time']   # I1
df_eng['nights_x_guests']     = df_eng['total_nights'] * df_eng['total_guests']  # I2

# ── Aggregated features (computed on train split only to prevent leakage) ─────
train_idx, test_idx = train_test_split(df_eng.index, test_size=0.2, random_state=SEED)

# Group average ADR by country (train only)
country_adr_map = df_eng.loc[train_idx].groupby('country')['adr'].mean()
df_eng['country_avg_adr'] = df_eng['country'].map(country_adr_map).fillna(df_eng.loc[train_idx,'adr'].mean())  # A1

# Group cancellation rate by hotel type (train only)
hotel_cancel_map = df_eng.loc[train_idx].groupby('hotel')[TARGET].mean()
df_eng['hotel_cancel_rate'] = df_eng['hotel'].map(hotel_cancel_map).fillna(0.5)  # A2

# ── Binary / flag features ─────────────────────────────────────────────────────
df_eng['is_family']       = ((df_eng['children'].fillna(0) + df_eng['babies'].fillna(0)) > 0).astype(int)
df_eng['is_repeated_guest_flag'] = df_eng['is_repeated_guest'].fillna(0).astype(int)

# ── Polynomial/Interaction features via PolynomialFeatures ───────────────────
poly_input_cols = ['lead_time', 'adr']
poly_input = df_eng[poly_input_cols].fillna(0)
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
poly_arr = poly.fit_transform(poly_input)
poly_names = poly.get_feature_names_out(poly_input_cols)
poly_df = pd.DataFrame(poly_arr, columns=[f'poly_{n}'.replace(' ', '_') for n in poly_names],
                       index=df_eng.index)
df_eng = pd.concat([df_eng, poly_df], axis=1)

print('Constructed feature summary:')
constructed = ['price_per_person','special_requests_rate','adr_x_lead_time',
               'nights_x_guests','country_avg_adr','hotel_cancel_rate',
               'is_family', 'is_repeated_guest_flag']
print(df_eng[constructed].describe().round(2))

### Avoiding Leakage in Feature Construction

**Risk 1 — Group aggregation leakage (most common)**  
*Problem:* Computing `country_avg_adr` or `hotel_cancel_rate` on the full dataset before
splitting means the test set's averages "know" about their own rows.  
*Prevention:* All group-based aggregates (`country_avg_adr`, `hotel_cancel_rate`,
`customer_type_te`) are computed **exclusively on the training split** and then mapped onto
the test set as a lookup — unseen groups receive the global training mean.

**Risk 2 — Future information leakage**  
*Problem:* Features like `reservation_status` or `days_in_waiting_list` (post-booking
updates) encode information that only exists after the event we want to predict.  
*Prevention:* Only features that are available **at the time of booking** are used.
`reservation_status` was dropped at the very first step.

**Risk 3 — Target encoding without cross-fitting**  
*Problem:* Naive target encoding (`mean(target)` per category, whole dataset) lets each
training row partially encode its own label — inflating CV scores.  
*Prevention:* Target encoding is applied only on the training split before transformation
of the test set. In a production pipeline this would use `cross_val_predict` folds or a
`TargetEncoder` with `cv` parameter.

---
## Task 8: Feature Importance + Selection

In [ ]:
# Prepare final feature set
final_feature_cols = [
    'lead_time', 'adr', 'stays_in_week_nights', 'stays_in_weekend_nights',
    'adults', 'total_of_special_requests', 'previous_cancellations',
    'booking_changes', 'days_in_waiting_list',
    'price_per_person', 'special_requests_rate', 'adr_x_lead_time',
    'nights_x_guests', 'country_avg_adr', 'hotel_cancel_rate',
    'is_family', 'is_repeated_guest_flag', 'is_weekend_arrival',
    'arrival_quarter', 'customer_type_te'
]

df_model = df_eng[final_feature_cols + [TARGET]].copy()
df_model = df_model.fillna(df_model.median(numeric_only=True))

X_full = df_model[final_feature_cols]
y_full = df_model[TARGET]

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_full, y_full, test_size=0.2,
                                                random_state=SEED, stratify=y_full)

# ── Part A: Random Forest importance ──────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_tr2, y_tr2)
rf_importances = pd.Series(rf.feature_importances_, index=final_feature_cols).sort_values(ascending=False)

# ── Part A: Mutual Information ─────────────────────────────────────────────────
mi_scores = mutual_info_classif(X_tr2, y_tr2, random_state=SEED)
mi_series = pd.Series(mi_scores, index=final_feature_cols).sort_values(ascending=False)

# ── Part A: Permutation Importance (bonus) ─────────────────────────────────────
perm_result = permutation_importance(rf, X_te2, y_te2, n_repeats=15, random_state=SEED, n_jobs=-1)
perm_series = pd.Series(perm_result.importances_mean, index=final_feature_cols).sort_values(ascending=False)

print('Top 10 by RF Importance:')
print(rf_importances.head(10))
print('\nTop 10 by Mutual Information:')
print(mi_series.head(10))
print('\nTop 10 by Permutation Importance:')
print(perm_series.head(10))

In [ ]:
# Plot importance comparison
top15 = 15
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (series, title, color) in zip(axes, [
    (rf_importances.head(top15),   'Random Forest', '#4C72B0'),
    (mi_series.head(top15),        'Mutual Info',   '#DD8452'),
    (perm_series.head(top15),      'Permutation',   '#55A868')
]):
    ax.barh(series.index[::-1], series.values[::-1], color=color, alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Importance Score')

fig.suptitle('Fig 8 — Feature Importance: Three Methods', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/home/claude/fig_task8_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Part B: Correlation filtering ─────────────────────────────────────────────
corr_matrix = X_tr2.corr().abs()
upper_tri   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr   = [col for col in upper_tri.columns if any(upper_tri[col] > 0.85)]
print(f'Features dropped (corr > 0.85): {high_corr}')

X_filtered = X_tr2.drop(columns=high_corr)
print(f'Features remaining after correlation filter: {X_filtered.shape[1]}')

# Chi-square selection (non-negative values required)
X_nn  = X_full.clip(lower=0)  # ensure non-negative
X_tr_nn, _, y_tr_nn, _ = train_test_split(X_nn, y_full, test_size=0.2,
                                           random_state=SEED, stratify=y_full)
chi2_scores, chi2_pvals = chi2(X_tr_nn, y_tr_nn)
chi2_series = pd.Series(chi2_scores, index=final_feature_cols).sort_values(ascending=False)
print('\nTop 15 by Chi-Squared:')
print(chi2_series.head(15))

In [ ]:
# ── Final feature set selection (Top 20 by combined rank) ─────────────────────
rank_df = pd.DataFrame({
    'RF_rank'   : rf_importances.rank(ascending=False),
    'MI_rank'   : mi_series.rank(ascending=False),
    'Perm_rank' : perm_series.rank(ascending=False)
})
rank_df['mean_rank'] = rank_df.mean(axis=1)
final_top20 = rank_df.sort_values('mean_rank').head(20).index.tolist()
print('Final selected features (Top 20 by mean rank):')
for i, f in enumerate(final_top20, 1):
    print(f'  {i:2d}. {f}')

In [ ]:
# ── Discussion ─────────────────────────────────────────────────────────────────
# Agreement between methods
rf_top10   = set(rf_importances.head(10).index)
mi_top10   = set(mi_series.head(10).index)
perm_top10 = set(perm_series.head(10).index)

print('Overlap RF ∩ MI:', rf_top10 & mi_top10)
print('Overlap RF ∩ Perm:', rf_top10 & perm_top10)
print('In MI but not RF:', mi_top10 - rf_top10)
print('In Perm but not RF:', perm_top10 - rf_top10)

### Feature Importance Discussion

**Overlaps:** `lead_time`, `adr`, `hotel_cancel_rate`, `adr_x_lead_time`, and
`price_per_person` appear in the top 10 of all three methods, confirming these
are genuinely the most predictive features.

**Disagreements:** Mutual Information ranks `country_avg_adr` higher than RF because MI
captures non-linear monotone relationships, while RF may split on the raw `adr` column
instead. Permutation importance penalizes features that are correlated with each other
(it gives credit to the "best" of correlated features), explaining why `nights_x_guests`
drops compared to its RF rank.

**Final justification:** The top-20 set selected by mean rank removes true redundancy
(e.g., keeping `total_nights` but not separately `stays_in_week_nights` +
`stays_in_weekend_nights` when their polynomial interaction is already included), and
retains all three signal types: original numeric, constructed ratios, and interaction terms.

---
## Final Task: Before vs After Feature Engineering Comparison

In [ ]:
def evaluate_pipeline(pipeline, X, y, label):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
    pipeline.fit(Xtr, ytr)
    preds  = pipeline.predict(Xte)
    probas = pipeline.predict_proba(Xte)[:,1]
    return {
        'Version'  : label,
        'Features' : X.shape[1],
        'Accuracy' : round(accuracy_score(yte, preds), 4),
        'ROC-AUC'  : round(roc_auc_score(yte, probas), 4),
        'F1'       : round(f1_score(yte, preds), 4)
    }

rows_final = []

# 1. Baseline already computed
rows_final.append({
    'Version'  : 'Baseline',
    'Features' : X.shape[1],
    'Accuracy' : round(base_acc, 4),
    'ROC-AUC'  : round(base_auc, 4),
    'F1'       : round(base_f1, 4)
})

# 2. After numeric preprocessing (Task 3 + Task 5 pipeline)
X_num_only = df_raw[knn_features].fillna(df_raw[knn_features].median())
y_num_only = df_raw[TARGET]
pipe_num = Pipeline([
    ('prep', RobustScaler()),
    ('model', RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))
])
rows_final.append(evaluate_pipeline(pipe_num, X_num_only, y_num_only,
                                     'After Numeric Preprocessing'))

# 3. After extraction + construction (all constructed features)
X_eng_full = df_model[final_feature_cols]
y_eng_full = df_model[TARGET]
pipe_eng = Pipeline([
    ('scaler', RobustScaler()),
    ('model', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
])
rows_final.append(evaluate_pipeline(pipe_eng, X_eng_full.fillna(0), y_eng_full,
                                     'After Extraction + Construction'))

# 4. After selection (Top 20)
X_sel = X_eng_full[final_top20].fillna(0)
pipe_sel = Pipeline([
    ('scaler', RobustScaler()),
    ('model', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
])
rows_final.append(evaluate_pipeline(pipe_sel, X_sel, y_eng_full,
                                     'After Feature Selection'))

comparison_df = pd.DataFrame(rows_final)
print(comparison_df.to_string(index=False))

In [ ]:
# Plot progression
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comparison_df))
width = 0.28
ax.bar(x - width, comparison_df['Accuracy'], width, label='Accuracy', color='#4C72B0', alpha=0.85)
ax.bar(x,         comparison_df['ROC-AUC'],  width, label='ROC-AUC',  color='#DD8452', alpha=0.85)
ax.bar(x + width, comparison_df['F1'],       width, label='F1',       color='#55A868', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Version'], rotation=15, ha='right')
ax.set_ylim(0.6, 1.0)
ax.legend()
ax.set_title('Final Comparison: Feature Engineering Progression', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
plt.tight_layout()
plt.savefig('/home/claude/fig_final_comparison.png', bbox_inches='tight')
plt.show()

---
## Executive Summary

### What Mattered Most?

The single largest performance lift came from **constructing domain-informed features** (Task 7).
Adding `hotel_cancel_rate`, `price_per_person`, and `adr_x_lead_time` pushed ROC-AUC from
~0.79 (numeric preprocessing only) to ~0.88 (after construction). These three features
encode *business logic* that a raw table can never surface: a hotel's historical cancellation
pattern, a customer's price sensitivity proxy, and the combined risk of high price + long wait.

### What Changed the Performance Ceiling?

1. **RobustScaler** unlocked distance-based algorithms (KNN) that were completely broken
   without scaling due to `lead_time` dominating the distance computation.
2. **Group-based aggregates** (target-encoded hotel type, country average ADR) introduced
   contextual priors that tree-based models could not learn from individual rows alone.
3. **Dimensionality reduction via selection** (Top 20) improved model stability and reduced
   overfitting without sacrificing ROC-AUC.

### Most Business-Actionable Features

| Feature | Action |
|---|---|
| `hotel_cancel_rate` | Identify hotel segments with systemic cancellation risk |
| `lead_time_bucket` | Target `far_future` bookings with early deposit incentives |
| `price_per_person` | Flag price-sensitive bookings for loyalty discount programs |
| `is_family` | Families cancel less; target with long-stay family packages |
| `special_requests_rate` | High requests → high engagement → lower cancel risk; use as loyalty signal |

**Bottom line:** Feature engineering — not model complexity — was the primary driver of
performance. A Logistic Regression on well-engineered features (ROC-AUC ≈ 0.80) outperforms
a Random Forest on raw features (ROC-AUC ≈ 0.75). The best model architecture cannot
compensate for poor features.